# 68 - `test23_june_3` Strict vs `apo_gate` ANG/PEN/FL Comparison

This notebook reconstructs the Notebook 67 `apo_gate` final ANG/PEN/FL series for `test23_june_3` and compares it against the existing strict-run outputs using the same three-panel style as `test23_june_3_ANG_PEN_FL_over_time.png`.

The full-sequence plot keeps y-axis limits anchored to the original strict data. If the experimental `apo_gate` variant blows up outside that range, the point is marked at the plot boundary instead of stretching the axis and hiding the clinically relevant jump.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.ultratimtrack_matlab_2state import MatlabTwoStateKalmanConfig, run_matlab_2state_kalman

DATASET = 'test23_june_3'
STRICT_DIR = ROOT / 'results' / 'strict_ultratimtrack_runs' / DATASET
NB67_DIR = ROOT / 'results' / 'notebook67_anatomical_gating_diagnostics' / DATASET
OUT_DIR = ROOT / 'results' / 'notebook68_apo_gate_comparison'
OUT_DIR.mkdir(parents=True, exist_ok=True)

STRICT_NPZ = STRICT_DIR / f'{DATASET}_strict_results.npz'
STRICT_METADATA = STRICT_DIR / f'{DATASET}_strict_metadata.json'
GATED_APO_NPZ = NB67_DIR / f'{DATASET}_gated_aponeurosis_log.npz'
UTT_EXPORT = Path('/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat')

for path in [STRICT_NPZ, STRICT_METADATA, GATED_APO_NPZ, UTT_EXPORT]:
    assert path.exists(), path

print('strict:', STRICT_NPZ)
print('apo_gate log:', GATED_APO_NPZ)
print('output:', OUT_DIR)


strict: /Users/grosbedou/PycharmProjects/NDORMS/results/strict_ultratimtrack_runs/test23_june_3/test23_june_3_strict_results.npz
apo_gate log: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_gated_aponeurosis_log.npz
output: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison


## 1. Load Strict Data and Reconstruct `apo_gate`

Notebook 67 saved the gated aponeurosis state, not the full final ANG/PEN/FL result. This cell rebuilds the exact `apo_gate` branch using:

- original strict KLT prior fascicle segments;
- original strict TimTrack alpha stream;
- Notebook 67 gated superficial/deep aponeurosis lines;
- original adaptive-R theta/length scales, with the same per-frame aponeurosis length-side penalty used in Notebook 67.


In [2]:
def load_npz(path: Path) -> dict[str, np.ndarray]:
    with np.load(path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}

strict = load_npz(STRICT_NPZ)
gated_apo = load_npz(GATED_APO_NPZ)
metadata = json.loads(STRICT_METADATA.read_text())
mat_root = loadmat(UTT_EXPORT, simplify_cells=True)['UTT_numeric_export']
r_values = np.asarray(mat_root.get('R', [3.05529211]), dtype=float).reshape(-1)

n = min(len(strict['frame']), len(gated_apo['deep_lines']))
mm_per_px = float(np.asarray(strict['mm_per_pixel']).reshape(-1)[0])
kalman_cfg = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root.get('Q', 0.01)),
    x_measurement_variance=float(mat_root.get('X', 100.0)),
    alpha_measurement_variance=float(r_values[0]),
    n_start_frames=int(mat_root.get('NS', 1)),
    run_smoother=True,
    use_adaptive_R=True,
)

base_r_scale = np.asarray(strict.get('r_scale', np.ones(n)), dtype=float)[:n]
base_theta = np.asarray(strict.get('r_scale_theta', np.ones(n)), dtype=float)[:n]
base_length = np.asarray(strict.get('r_scale_length', np.ones(n)), dtype=float)[:n]
length_extra = np.maximum(1.0, np.nanmax(np.asarray(gated_apo['r_scales'], dtype=float)[:n], axis=1))
length_extra = np.clip(length_extra, 1.0, 50.0)

apo_gate = run_matlab_2state_kalman(
    strict['klt_prior_segments'][:n],
    strict['timtrack_alpha_deg'][:n],
    gated_apo['super_lines'][:n],
    gated_apo['deep_lines'][:n],
    config=kalman_cfg,
    mm_per_pixel=mm_per_px,
    measurement_r_scale=base_r_scale,
    measurement_r_scale_theta=base_theta,
    measurement_r_scale_length=np.maximum(base_length, length_extra),
)

series = pd.DataFrame({
    'Frame': strict['frame'][:n],
    'Time': strict['time_s'][:n],
    'ANG_normal_Kalman': strict['fixed_ANG_deg'][:n],
    'ANG_strict_selected': strict['ANG_deg'][:n],
    'ANG_apo_gate': apo_gate['ANG_deg'][:n],
    'PEN_normal_Kalman': strict['fixed_PEN_deg'][:n],
    'PEN_strict_selected': strict['PEN_deg'][:n],
    'PEN_apo_gate': apo_gate['PEN_deg'][:n],
    'FL_normal_Kalman_mm': strict['fixed_FL_mm'][:n],
    'FL_strict_selected_mm': strict['FL_mm'][:n],
    'FL_apo_gate_mm': apo_gate['FL_mm'][:n],
    'deep_line_rejected': np.asarray(gated_apo['line_rejected'][:n, 1], dtype=bool),
})
series_path = OUT_DIR / f'{DATASET}_strict_vs_apo_gate_series.csv'
series.to_csv(series_path, index=False)
series.head(), series_path


(   Frame      Time  ANG_normal_Kalman  ANG_strict_selected  ANG_apo_gate  \
 0      0  0.000000          20.800000            20.800000     20.800000   
 1      1  0.052632          20.774252            20.774196     20.774196   
 2      2  0.105263          20.767022            20.766962     20.766962   
 3      3  0.157895          20.765821            20.765761     20.765761   
 4      4  0.210526          20.773274            20.773210     20.773210   
 
    PEN_normal_Kalman  PEN_strict_selected  PEN_apo_gate  FL_normal_Kalman_mm  \
 0          14.469660            14.469660     14.469660            75.328253   
 1          14.507382            14.507327     14.507327            75.206955   
 2          14.519803            14.519743     14.519743            75.096098   
 3          14.496952            14.496892     14.496892            75.197716   
 4          14.321992            14.321927     14.321927            75.556157   
 
    FL_strict_selected_mm  FL_apo_gate_mm  deep_

## 2. Plot Helpers


In [3]:
def baseline_ylim(*arrays: np.ndarray, pad_fraction: float = 0.08) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a, dtype=float).reshape(-1) for a in arrays])
    vals = vals[np.isfinite(vals)]
    lo = float(np.min(vals))
    hi = float(np.max(vals))
    pad = max((hi - lo) * pad_fraction, 0.5)
    return lo - pad, hi + pad


def plot_with_boundary_markers(ax, x, y, *, ylim, color, label, linestyle='-', linewidth=1.3):
    x = np.asarray(x)
    y = np.asarray(y, dtype=float)
    lo, hi = ylim
    in_view = np.isfinite(y) & (y >= lo) & (y <= hi)
    y_view = y.copy()
    y_view[~in_view] = np.nan
    ax.plot(x, y_view, color=color, linestyle=linestyle, linewidth=linewidth, label=label)
    high = np.isfinite(y) & (y > hi)
    low = np.isfinite(y) & (y < lo)
    if np.any(high):
        ax.scatter(x[high], np.full(int(high.sum()), hi), marker='^', color=color, s=18, zorder=5, label=f'{label} out of view ({int(high.sum())})')
    if np.any(low):
        ax.scatter(x[low], np.full(int(low.sum()), lo), marker='v', color=color, s=18, zorder=5)


def make_three_panel_plot(*, zoom: bool = False) -> Path:
    if zoom:
        frame_mask = (series['Frame'] >= 84) & (series['Frame'] <= 96)
        suffix = 'problem_window_ANG_PEN_FL'
        title = f'{DATASET}: strict vs apo_gate, frames 84-96'
    else:
        frame_mask = np.ones(len(series), dtype=bool)
        suffix = 'ANG_PEN_FL_over_time'
        title = f'{DATASET}: strict vs apo_gate outputs over time'

    df = series.loc[frame_mask].copy()
    time = df['Time'].to_numpy()
    fig, axes = plt.subplots(3, 1, figsize=(13.5, 7.8), sharex=True)

    specs = [
        {
            'ax': axes[0],
            'ylabel': 'ANG (deg)',
            'normal': 'ANG_normal_Kalman',
            'strict': 'ANG_strict_selected',
            'apo': 'ANG_apo_gate',
            'color': 'tab:red',
            'legend_loc': 'upper right',
        },
        {
            'ax': axes[1],
            'ylabel': 'PEN (deg)',
            'normal': 'PEN_normal_Kalman',
            'strict': 'PEN_strict_selected',
            'apo': 'PEN_apo_gate',
            'color': 'tab:blue',
            'legend_loc': 'upper right',
        },
        {
            'ax': axes[2],
            'ylabel': 'FL (mm)',
            'normal': 'FL_normal_Kalman_mm',
            'strict': 'FL_strict_selected_mm',
            'apo': 'FL_apo_gate_mm',
            'color': 'tab:green',
            'legend_loc': 'upper left',
        },
    ]

    for spec in specs:
        ax = spec['ax']
        ylim = baseline_ylim(df[spec['normal']].to_numpy(), df[spec['strict']].to_numpy())
        if zoom:
            ylim = baseline_ylim(df[spec['normal']].to_numpy(), df[spec['strict']].to_numpy(), df[spec['apo']].to_numpy(), pad_fraction=0.10)
        ax.plot(time, df[spec['normal']], color='black', linewidth=1.2, label='normal Kalman')
        ax.plot(time, df[spec['strict']], color=spec['color'], linewidth=1.4, label='strict selected Kalman')
        plot_with_boundary_markers(
            ax,
            time,
            df[spec['apo']],
            ylim=ylim,
            color=spec['color'],
            label='apo_gate modified',
            linestyle='--',
            linewidth=1.3,
        )
        ax.set_ylim(*ylim)
        ax.set_ylabel(spec['ylabel'])
        ax.grid(True, alpha=0.28)
        ax.legend(loc=spec['legend_loc'], fontsize=9)

    if zoom:
        reject_times = df.loc[df['deep_line_rejected'], 'Time'].to_numpy()
        for ax in axes:
            for t in reject_times:
                ax.axvline(t, color='0.2', alpha=0.12, linewidth=1.0)
    else:
        problem = series[(series['Frame'] >= 88) & (series['Frame'] <= 91)]
        for ax in axes:
            ax.axvspan(problem['Time'].min(), problem['Time'].max(), color='gold', alpha=0.14, linewidth=0)

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title, fontsize=15)
    fig.tight_layout()
    out = OUT_DIR / f'{DATASET}_strict_vs_apo_gate_{suffix}.png'
    fig.savefig(out, dpi=160)
    plt.close(fig)
    return out


## 3. Same-Style Full-Time Plot and Problem-Window Plot

The full-time figure has the same three panels as the attached strict-run graph. The `apo_gate` line is dashed. Values outside the original strict y-range are marked at the boundary instead of stretching the axis.


In [4]:
full_plot = make_three_panel_plot(zoom=False)
zoom_plot = make_three_panel_plot(zoom=True)
full_plot, zoom_plot


(PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_strict_vs_apo_gate_ANG_PEN_FL_over_time.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_strict_vs_apo_gate_problem_window_ANG_PEN_FL.png'))

## 4. Numeric Check Around the Reported Jump


In [5]:
window = series[(series['Frame'] >= 84) & (series['Frame'] <= 96)].copy()
summary = pd.DataFrame([
    {
        'series': 'strict selected',
        'FL_range_88_91_mm': window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_strict_selected_mm'].max() - window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_strict_selected_mm'].min(),
        'max_abs_FL_step_88_91_mm': window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_strict_selected_mm'].diff().abs().max(),
    },
    {
        'series': 'apo_gate modified',
        'FL_range_88_91_mm': window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_apo_gate_mm'].max() - window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_apo_gate_mm'].min(),
        'max_abs_FL_step_88_91_mm': window.loc[(window.Frame >= 88) & (window.Frame <= 91), 'FL_apo_gate_mm'].diff().abs().max(),
    },
])
summary_path = OUT_DIR / f'{DATASET}_frame_88_91_summary.csv'
summary.to_csv(summary_path, index=False)
window_path = OUT_DIR / f'{DATASET}_frames_84_96_strict_vs_apo_gate.csv'
window.to_csv(window_path, index=False)
summary, summary_path, window_path


(              series  FL_range_88_91_mm  max_abs_FL_step_88_91_mm
 0    strict selected          13.114962                 10.489112
 1  apo_gate modified           3.465910                  2.841916,
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_frame_88_91_summary.csv'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_frames_84_96_strict_vs_apo_gate.csv'))

In [6]:
print('Saved full-time plot:', full_plot)
print('Saved problem-window plot:', zoom_plot)
print('Saved series CSV:', series_path)
print('Saved summary CSV:', summary_path)
print('Saved window CSV:', window_path)


Saved full-time plot: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_strict_vs_apo_gate_ANG_PEN_FL_over_time.png
Saved problem-window plot: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_strict_vs_apo_gate_problem_window_ANG_PEN_FL.png
Saved series CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_strict_vs_apo_gate_series.csv
Saved summary CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_frame_88_91_summary.csv
Saved window CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook68_apo_gate_comparison/test23_june_3_frames_84_96_strict_vs_apo_gate.csv
